In [2]:
import random


class Server:
    def __init__(self, name, capacity):
        self.name = name
        self.capacity = capacity
        self.load = 0

    def available_capacity(self):
        return self.capacity - self.load

    def __str__(self):
        return f"{self.name}: Load={self.load}/{self.capacity}"


class Heuristic:

    @staticmethod
    def least_loaded(servers):
        return min(servers, key=lambda s: s.load)

    @staticmethod
    def capacity_proportional(servers):
        weights = [s.capacity for s in servers]
        return random.choices(servers, weights=weights, k=1)[0]


class HeuristicSelector:

    @staticmethod
    def select(servers):
        total_load = sum(s.load for s in servers)
        total_capacity = sum(s.capacity for s in servers)
        load_ratio = total_load / total_capacity

        if load_ratio < 0.5:
            return Heuristic.least_loaded
        else:
            return Heuristic.capacity_proportional


class CSPRepair:

    @staticmethod
    def repair(servers):
        for server in servers:
            if server.load > server.capacity:
                excess = server.load - server.capacity
                server.load = server.capacity

                for target in servers:
                    if target != server and target.available_capacity() > 0:
                        move = min(excess, target.available_capacity())
                        target.load += move
                        excess -= move
                        if excess == 0:
                            break


class ACO:

    def __init__(self, servers):
        self.pheromones = {s.name: 1.0 for s in servers}

    def choose_server(self, servers):
        weights = [
            self.pheromones[s.name] * max(s.available_capacity(), 0)
            for s in servers
        ]

        if sum(weights) == 0:
            return None

        return random.choices(servers, weights=weights, k=1)[0]

    def update_pheromones(self, servers, evaporation=0.1):
        for s in servers:
            self.pheromones[s.name] *= (1 - evaporation)
            if s.available_capacity() > 0:
                self.pheromones[s.name] += 0.3


def dispatch(server):
    if server and server.available_capacity() > 0:
        server.load += 1
        return True
    return False


def log_state(round_num, servers):
    print(f"\nRound {round_num}")
    for s in servers:
        print(s)


def simulate():
    servers = [
        Server("S1", 10),
        Server("S2", 15),
        Server("S3", 20)
    ]

    aco = ACO(servers)

    rounds = 5
    requests_per_round = 15

    for r in range(1, rounds + 1):

        for _ in range(requests_per_round):

            if all(s.available_capacity() == 0 for s in servers):
                continue

            heuristic = HeuristicSelector.select(servers)
            heuristic_server = heuristic(servers)

            aco_server = aco.choose_server(servers)
            chosen_server = aco_server if aco_server else heuristic_server

            dispatch(chosen_server)

        CSPRepair.repair(servers)
        aco.update_pheromones(servers)
        log_state(r, servers)


if __name__ == "__main__":
    simulate()


Round 1
S1: Load=4/10
S2: Load=5/15
S3: Load=6/20

Round 2
S1: Load=8/10
S2: Load=10/15
S3: Load=12/20

Round 3
S1: Load=10/10
S2: Load=15/15
S3: Load=20/20

Round 4
S1: Load=10/10
S2: Load=15/15
S3: Load=20/20

Round 5
S1: Load=10/10
S2: Load=15/15
S3: Load=20/20
